# Kaggriculture | From Orders to Actual Trades

**Reconstruct what actually traded and where the money went.**

This notebook replays Kaggriculture's market settlement one unit at a time, accounting
for partial fills, changing prices, shed capacity, and available cash. It turns a replay
into a revenue-and-cost ledger, then checks that ledger against both players' recorded
cash balances on every turn.

Why reconstruct the trades? A bot can request a sale of 100 units with only three in
its shed. A purchase can stop when cash runs out. And within a single order, successive
units can trade at different prices. The ledger follows those fills through settlement.

You get:

- Sales revenue, units sold, and realized average prices for each product.
- Spending on seeds, livestock, feed, fertilizer, labor, and land.
- Daily cash-flow charts and a breakdown of spending by category.
- Turn-by-turn balance checks that flag unexplained changes in cash.

During play, cash changes through `SELL`, `BUY_SEED`, `BUY_PRODUCT`, `BUY_ANIMAL`,
`HIRE`, and `BUY_LAND`. Farmer and hand actions run first, so the reconstruction also
applies their changes before processing the market queue.

The demo runs a simple farmer against the built-in starter. See sections 5 and 6 for
the accounting results, or section 8 to analyse your own replay. Full private
observations for both players are required.


In [ ]:
%%capture
!pip install "kaggle-environments==1.32.7"

In [ ]:
import warnings
from collections import defaultdict
from copy import deepcopy

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

import kaggle_environments
from kaggle_environments import make

try:
    from kaggle_environments.envs.kaggriculture.kaggriculture import (
        MARKET_PARAMS, CROPS, ANIMALS, PRODUCTS, LAND_PRICES, PRICE_FLOOR, market_price,
        _apply_unit_action,
    )
except ImportError as exc:                 # Check that the installed engine provides the required helpers.
    raise ImportError(
        f"{exc} -- this notebook needs kaggle-environments == 1.32.7, and the installed "
        f"version is {kaggle_environments.__version__}. Turn on internet so the install "
        "cell above can run, or upgrade the package."
    ) from exc

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)

# --- a consistent style for the figures below
SURFACE, INK, INK_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#dcdbd6"
BLUE, ORANGE, RED = "#2a78d6", "#eb6834", "#e34948"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 130, "font.size": 9,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.titlepad": 10, "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": INK_2, "ytick.color": INK_2, "text.color": INK,
    "grid.color": GRID, "grid.linewidth": 0.6, "legend.frameon": False,
})

print("kaggle_environments", kaggle_environments.__version__)
print("market parameters loaded for:", ", ".join(PRODUCTS))

## 1. The price curve

For a given set of market parameters, a product's sell quote depends on its current
market inventory. Under the default settings, every product starts at `I0 = 10,000`.
Sales above the $1 floor add stock; town consumption and purchases remove it. Prices
fall as inventory rises and increase as it falls, subject to rounding and the floor.

We use `market_price` from the installed engine. Replay analysis merges any
`marketParams` overrides with that version's defaults. The table below shows the
**default parameters**, independently of the demo episode.

In [ ]:
def price_of(item, inventory, params=MARKET_PARAMS):
    """Return the installed engine's sell quote at the specified inventory.

    Pass the episode's resolved parameters when analysing a replay. Prices are
    rounded to whole dollars and cannot fall below the engine's price floor.
    """
    return market_price(item, inventory, params)


summary = pd.DataFrame([{
    "product": item,
    "base": p["base"],
    "T (one field, 24 days)": p["T"],
    "price at I0+T": price_of(item, p["I0"] + p["T"]),
    "price at I0+2T": price_of(item, p["I0"] + 2 * p["T"]),
} for item, p in MARKET_PARAMS.items()])
summary["price at I0+T (% of base)"] = (summary["price at I0+T"] / summary["base"]).map("{:.0%}".format)
summary

`T` is a reference production quantity for a 5x5 field over 24 days, assuming optimal
watering and no fertilizer. The engine discounts animal quantities for feed overhead.
The `I0+T` column is the sell quote **at that inventory**, rather than the average
price received for selling `T` units.

Before the floor is reached, the T-th unit sold from `I0` is quoted at `I0+T-1`.
Floor-price sales add no inventory, so an actual sequence of sales may never reach
`I0+T`.

At `I0+T`, wheat and eggs retain about 80% of their base price; melon, strawberry,
milk, and wool are at the $1 floor. Section 7 compares the default curves in more detail.

## 2. A demo season

The demo farmer gives the accounting examples a mix of expenses. It buys seeds,
raises a goose, purchases another quadrant, and hires a hand on mornings 2 through 26
when it has enough cash. Its movement and work priorities are deliberately simple.

This example illustrates the ledger; its score is not a benchmark for competitive play.

In [ ]:
CROP_PLAN = ["CARROT", "WHEAT", "TOMATO"]
SELLABLE = ["CARROT", "TOMATO", "EGG", "FERTILIZER"]   # Reserve wheat for feed.
COOP_XY = (0, 0)
SHED_TILES = {(4, 4), (5, 4), (4, 5), (5, 5)}
P_ANIMAL, P_HARVEST, P_WATER, P_WEED, P_PLANT = 0, 1, 2, 3, 4   # Lower values have higher priority.


def _ripe(tile, day):
    if tile.get("yield_units", 0) <= 0:
        return False
    c = CROPS[tile["crop"]]
    return c["ongoing"] or day - tile["planted_day"] >= c["max_yield_day"]


def _tile_job(pos, farm, private, day, inv, want_coop):
    """Return the priority and action for a unit at `pos`, if work is available."""
    x, y = pos
    tile = farm["tiles"][y][x]
    if isinstance(tile, dict):
        kind = tile.get("kind")
        if kind == "PLANT":
            if _ripe(tile, day):                       return P_HARVEST, ["HARVEST"]
            if not tile["watered_today"]:              return P_WATER, ["WATER"]
        elif kind == "WEED":                           return P_WEED, ["DIG"]
        elif kind in ("COOP", "PASTURE"):
            if tile.get("animal") is None:
                if inv.get("GOOSE", 0) > 0:            return P_ANIMAL, ["PLACE", "GOOSE"]
            else:
                if not tile["fed_today"] and inv.get("WHEAT", 0) > 0:
                    return P_ANIMAL, ["FEED"]
                if tile.get("yield_units", 0) > 0:     return P_ANIMAL, ["HARVEST"]
                if tile.get("fertilizer_available"):   return P_ANIMAL, ["COLLECT_FERTILIZER"]
                if not tile.get("cared_today"):        return P_ANIMAL, ["CARE"]
    elif tile is None:
        if want_coop and (x, y) == COOP_XY:            return P_ANIMAL, ["BUILD_COOP"]
        for crop in CROP_PLAN:
            if private.get("seeds", {}).get(crop, 0) > 0:
                return P_PLANT, ["PLANT", crop]
    return None


def _step_toward(pos, target):
    x, y = pos
    tx, ty = target
    if x < tx: return ["EAST"]
    if x > tx: return ["WEST"]
    if y < ty: return ["SOUTH"]
    if y > ty: return ["NORTH"]
    return ["PASS"]


def make_demo_agent(batch=3):
    """Create a demo farmer that sells up to `batch` units per product per turn."""

    def agent(obs):
        me = obs["player"]
        farm, private = obs["farms"][me], obs["private"]
        day, hour = obs["day"], obs["hour"]
        shed, seeds, money = private.get("shed", {}), private.get("seeds", {}), farm["money"]
        invs = private.get("inventories", [{}]) or [{}]
        tiles = farm["tiles"]
        coop = tiles[COOP_XY[1]][COOP_XY[0]]
        coop_ready = isinstance(coop, dict) and coop.get("kind") == "COOP"
        want_coop = not coop_ready
        has_goose = coop_ready and coop.get("animal") == "GOOSE"
        goose_incoming = shed.get("GOOSE", 0) > 0 or any(i.get("GOOSE", 0) for i in invs)
        needs_feed = has_goose and not coop.get("fed_today", False)

        # ---- market orders: the only channel that moves money
        market = []
        for crop in CROP_PLAN:
            if seeds.get(crop, 0) < 3 and money > 300:
                market.append(["BUY_SEED", crop, 3])
        if coop_ready and not has_goose and not goose_incoming and money > 900:
            market.append(["BUY_ANIMAL", "GOOSE", 1])
        if has_goose and shed.get("WHEAT", 0) + sum(i.get("WHEAT", 0) for i in invs) < 2 and money > 200:
            market.append(["BUY_PRODUCT", "WHEAT", 2])
        if day in (8, 14) and hour == 2 and money > 800:
            market.append(["BUY_PRODUCT", "FERTILIZER", 1])
        if day >= 6 and len(farm["unlocked_quadrants"]) == 1 and money >= 2500:
            market.append(["BUY_LAND"])
        if 2 <= day <= 26 and hour == 0 and money > 100:
            market.append(["HIRE"])
        for item in SELLABLE:
            have = shed.get(item, 0)
            if have > 0:
                market.append(["SELL", item, min(batch, have)])
        market = market[:10]

        # ---- unit routing
        def act_for(idx, pos):
            inv = invs[idx] if idx < len(invs) else {}
            here = _tile_job(pos, farm, private, day, inv, want_coop and idx == 0)
            if here:
                return here[1]
            if coop_ready and (inv.get("GOOSE", 0) > 0 or (needs_feed and inv.get("WHEAT", 0) > 0)):
                return _step_toward(pos, COOP_XY)
            if tuple(pos) in SHED_TILES:
                if not has_goose and shed.get("GOOSE", 0) > 0 and inv.get("GOOSE", 0) == 0:
                    return ["PICKUP", "GOOSE", 1]
                if needs_feed and inv.get("WHEAT", 0) == 0 and shed.get("WHEAT", 0) > 0:
                    return ["PICKUP", "WHEAT", 1]
                if sum(inv.values()) > 0:
                    return ["DROP"]
            if not has_goose and shed.get("GOOSE", 0) > 0:
                return _step_toward(pos, (4, 4))
            if needs_feed and not any(i.get("WHEAT", 0) for i in invs) and shed.get("WHEAT", 0) > 0:
                return _step_toward(pos, (4, 4))
            if sum(inv.values()) >= 12:
                return _step_toward(pos, (4, 4))
            jobs = []
            for y, row in enumerate(tiles):
                for x, tile in enumerate(row):
                    if tile == "LOCKED":
                        continue
                    job = _tile_job((x, y), farm, private, day, inv, want_coop and idx == 0)
                    if job:
                        jobs.append((job[0], abs(x - pos[0]) + abs(y - pos[1]), (x, y)))
            if not jobs:
                return _step_toward(pos, (4, 4))
            jobs.sort(key=lambda j: (j[0], j[1]) if idx == 0 else (j[0], -j[1]))
            return _step_toward(pos, jobs[0][2])

        return {"farmer": act_for(0, farm["farmer"]),
                "hands": [act_for(i + 1, h) for i, h in enumerate(farm.get("hands", []))],
                "market": market}

    return agent


SEED = 3
env = make("kaggriculture", configuration={"seed": SEED})
env.run([make_demo_agent(), "starter"])
episode = env.toJSON()

final = episode["steps"][-1][0]["observation"]["farms"]
print(f"demo farmer  ${final[0]['money']:,.0f}     built-in starter  ${final[1]['money']:,.0f}")

## 3. Check prices in the demo episode

Each observation records market inventory and sell quotes. We recompute every quote
using the installed engine's price function and the episode's resolved parameters,
then compare the result with the recorded value.

This checks the price inputs before we reconstruct the trades. On an episode generated
locally it is a formality, since the same engine wrote the quotes. It matters on a
downloaded replay, where a different engine version or overridden market parameters would
show up here. Snapshot quotes describe the market at that moment; later units in an order
may trade at different prices.


In [ ]:
def resolve_market_params(config):
    """Episodes may override any subset of a product's market parameters."""
    resolved = {item: dict(p) for item, p in MARKET_PARAMS.items()}
    for item, patch in ((config or {}).get("marketParams") or {}).items():
        if item in resolved and isinstance(patch, dict):
            resolved[item].update(patch)
    return resolved


CONFIG = episode.get("configuration") or {}
PARAMS = resolve_market_params(CONFIG)

steps = episode["steps"]
checked = mismatched = 0
for st in steps:
    mk = st[0]["observation"]["market"]
    for item in PRODUCTS:
        checked += 1
        mismatched += price_of(item, mk["inventory"][item], PARAMS) != mk["prices"][item]

print(f"{checked:,} quotes checked   {mismatched} mismatches")
assert mismatched == 0, "price model disagrees with the engine - do not trust the P&L below"

The next table describes the market conditions in this demo: each product's lowest,
highest, and final recorded quote, plus the fraction of observations above its base
price. It includes products neither player sold. These quotes provide context;
realized revenue comes from the fills reconstructed in sections 4 and 5.


In [ ]:
def price_path(item):
    """Every price the engine quoted for one product, in order."""
    return [st[0]["observation"]["market"]["prices"][item] for st in steps]


def market_row(item):
    path, base = price_path(item), PARAMS[item]["base"]
    return {
        "product": item,
        "base": base,
        "lowest": min(path),
        "highest": max(path),
        "final": path[-1],
        "observations above base (%)": "{:.1%}".format(sum(p > base for p in path) / len(path)),
    }


market_reality = pd.DataFrame([market_row(item) for item in PRODUCTS])
market_reality

In this demo, most products were quoted above base for much of the season. The farms
supplied relatively little while the town continued to consume stock; purchases also
reduced wheat inventory. Fertilizer ended below base.

Production, purchases, and shop openings shape each episode's price path. The next
step is to follow both players' orders through that market and calculate what they
actually earned and spent.


## 4. Replaying the market queue

The engine processes corresponding positions in both players' order lists together.
Within each position, it quotes one unit for each eligible order from the same market
snapshot, attempts both fills, and then quotes the next units from the updated inventory.

Three details matter:

- A sale uses the quote before adding the unit. A product purchase uses the quote at
  inventory minus one. A completed buy-and-resell sequence has zero net cash flow
  when no other trades or town consumption intervene.
- Sales at the $1 floor do not add market inventory.
- When an order can no longer fill, its remaining quantity is abandoned. Later orders
  keep their original positions in the queue.

`HIRE` and `BUY_LAND` run once at their queue position, before that position's unit loop.
Shed quantities, capacity, and available cash determine which trades can fill.

In [ ]:
BUYABLE = ("WHEAT", "FERTILIZER")
CATEGORY = {"SELL": "revenue", "BUY_SEED": "seeds", "BUY_ANIMAL": "livestock",
            "HIRE": "labor", "BUY_LAND": "land"}


def fib(n):
    a, b = 1, 1
    for _ in range(n):
        a, b = b, a + b
    return a


def order_category(op, item):
    if op == "BUY_PRODUCT":
        return "fertilizer" if item == "FERTILIZER" else "feed"
    return CATEGORY.get(op, "other")


def parse_order(order):
    if not isinstance(order, list) or not order:
        return None
    op = order[0]
    if op in ("HIRE", "BUY_LAND"):
        return {"type": op}
    if op in ("BUY_SEED", "BUY_PRODUCT", "BUY_ANIMAL", "SELL") and len(order) >= 3:
        try:
            n = int(order[2])
        except (TypeError, ValueError):
            return None
        return {"type": op, "item": order[1], "remaining": n} if n > 0 else None
    return None


def replay_market_turn(queues, inventory, money, hires_today, quadrants, sheds,
                       params=MARKET_PARAMS, shed_capacity=100, hire_mult=1):
    """Re-run one turn of the market queue the way the engine does.

    `sheds` is what each player's shed holds when the queue starts, which gates what
    can fill. An order that cannot fill is skipped and abandoned in place, so every
    later order stays at its own index.

    Returns per-player (category, item, units, cash) rows plus the resulting market
    inventory, bank balances and shed contents.
    """
    inv, money = dict(inventory), list(money)
    hires, quads = list(hires_today), list(quadrants)
    shed = [dict(s or {}) for s in sheds]
    fills = [defaultdict(lambda: [0, 0.0]), defaultdict(lambda: [0, 0.0])]
    states = [[parse_order(o) for o in q] for q in queues]

    for i in range(max((len(s) for s in states), default=0)):
        slot = [s[i] if i < len(s) else None for s in states]

        for p, st in enumerate(slot):                       # Atomic orders at this queue position run first.
            if st is None:
                continue
            if st["type"] == "HIRE":
                cost = hire_mult * fib(hires[p])
                if money[p] >= cost:
                    money[p] -= cost
                    hires[p] += 1
                    fills[p][("labor", "HAND")][0] += 1
                    fills[p][("labor", "HAND")][1] -= cost
                slot[p] = None
            elif st["type"] == "BUY_LAND":
                extra = quads[p] - 1
                if extra < len(LAND_PRICES) and money[p] >= LAND_PRICES[extra]:
                    money[p] -= LAND_PRICES[extra]
                    quads[p] += 1
                    fills[p][("land", "QUADRANT")][0] += 1
                    fills[p][("land", "QUADRANT")][1] -= LAND_PRICES[extra]
                slot[p] = None

        for _ in range(99_999):                              # Match the engine's per-slot iteration limit.
            quoted = [None, None]
            for p, st in enumerate(slot):
                if st is None or st["remaining"] <= 0:
                    continue
                op, item = st["type"], st["item"]
                if op == "SELL" and item in PRODUCTS:
                    quoted[p] = (op, item, price_of(item, inv[item], params))
                elif op == "BUY_PRODUCT" and item in BUYABLE:
                    quoted[p] = (op, item, price_of(item, inv[item] - 1, params))
                elif op == "BUY_SEED" and item in CROPS:
                    quoted[p] = (op, item, CROPS[item]["seed"])
                elif op == "BUY_ANIMAL" and item in ANIMALS:
                    quoted[p] = (op, item, ANIMALS[item]["cost"])
                else:
                    slot[p] = None
            if all(q is None for q in quoted):
                break

            committed = False
            for p, q in enumerate(quoted):
                if q is None:
                    continue
                op, item, price = q
                if op == "SELL":
                    if shed[p].get(item, 0) <= 0:
                        slot[p] = None                       # nothing left to sell
                        continue
                    shed[p][item] = max(0, shed[p].get(item, 0) - 1)
                    money[p] += price
                    if price > PRICE_FLOOR:                  # floor sales add no supply
                        inv[item] += 1
                    cash = price
                else:
                    if money[p] < price:
                        slot[p] = None                       # out of money
                        continue
                    if (op in ("BUY_PRODUCT", "BUY_ANIMAL")
                            and sum(shed[p].values()) >= shed_capacity):
                        slot[p] = None                       # shed is full
                        continue
                    money[p] -= price
                    if op == "BUY_PRODUCT":
                        inv[item] -= 1
                    if op in ("BUY_PRODUCT", "BUY_ANIMAL"):
                        shed[p][item] = shed[p].get(item, 0) + 1
                    cash = -price
                cat = order_category(op, item)
                fills[p][(cat, item)][0] += 1
                fills[p][(cat, item)][1] += cash
                slot[p]["remaining"] -= 1
                committed = True
            if not committed:
                break

    ledger = [[(c, i, u, cash) for (c, i), (u, cash) in sorted(f.items())] for f in fills]
    return ledger, inv, money, shed

First, check a fertilizer purchase followed by a resale with no intervening market
activity. Then check that an empty shed blocks a sale and a full shed blocks a purchase.

In [ ]:
inv0 = {item: p["I0"] for item, p in PARAMS.items()}
empty = [{}, {}]
_, inv1, money1, shed1 = replay_market_turn(
    [[["BUY_PRODUCT", "FERTILIZER", 4]], []], inv0, [3000, 3000], [0, 0], [1, 1], empty, PARAMS)
_, inv2, money2, _ = replay_market_turn(
    [[["SELL", "FERTILIZER", 4]], []], inv1, money1, [0, 0], [1, 1], shed1, PARAMS)

print(f"buy  4 fertilizer: {money1[0] - 3000:+.0f}"
      f"   market inventory {inv0['FERTILIZER']} -> {inv1['FERTILIZER']}")
print(f"sell 4 fertilizer: {money2[0] - money1[0]:+.0f}"
      f"   market inventory {inv1['FERTILIZER']} -> {inv2['FERTILIZER']}")
print(f"net:               {money2[0] - 3000:+.0f}")

# The shed gates fills, so the replay has to gate them too.
rows_a, _, money_a, _ = replay_market_turn(
    [[["SELL", "CARROT", 5]], []], inv0, [3000, 3000], [0, 0], [1, 1], empty, PARAMS)
rows_b, _, money_b, _ = replay_market_turn(
    [[["BUY_ANIMAL", "GOOSE", 1]], []], inv0, [3000, 3000], [0, 0], [1, 1],
    [{"WHEAT": 100}, {}], PARAMS, shed_capacity=100)
print(f"sell 5 carrots, empty shed: {money_a[0] - 3000:+.0f}   rows={rows_a[0]}")
print(f"buy 1 goose, full shed:     {money_b[0] - 3000:+.0f}   rows={rows_b[0]}")

## 5. Reconstruct the cash flows

The key input is the shed **when the market starts**, not just the shed in the previous
observation. Unit actions run first. We copy each player's previous farm and private
state, then apply the farmer's action followed by the hands' actions using the installed
engine. This includes changes within a turn, such as a hand placing an animal in a
structure the farmer has just built. The original replay is not modified.

We then replay both players' recorded market orders, recording quantities and cash
only when units fill. This distinguishes a large request that partly filled from a
large sale, and a rejected purchase from an expense.

Finally, we compare the reconstructed ending balances with the recorded balances on
each turn. The report includes signed error, total absolute error, the largest
absolute error on one turn, and the number of player-turns with a mismatch. Positive
and negative errors can cancel in the signed total, so the absolute errors matter too.


In [ ]:
LEDGER_COLUMNS = ["step", "day", "player", "category", "item", "units", "cash"]


def shed_before_market(prev_cell, action, player, board_size=10, shed_capacity=100,
                       turns_per_day=24):
    """Reconstruct the shed after unit actions, without changing the recorded replay.

    Apply the farmer's action first, then each hand's action using the installed
    engine. Shared tile changes matter: a hand can place an animal in a structure
    built earlier in the same turn.
    """
    obs = prev_cell.get("observation") if isinstance(prev_cell, dict) else None
    private = obs.get("private") if isinstance(obs, dict) else None
    if not isinstance(private, dict) or any(
            key not in private for key in ("shed", "seeds", "inventories")):
        raise ValueError(f"Player {player}: replay is missing the full private state.")
    farms = obs.get("farms") if isinstance(obs, dict) else None
    farm = farms[player] if isinstance(farms, list) and player < len(farms) else None
    if not isinstance(farm, dict) or "tiles" not in farm:
        raise ValueError(
            f"Player {player}: this step carries no tile grid. Unit actions are replayed "
            "against the farm, so a replay trimmed to save space cannot be analyzed. "
            "Re-download the full episode, or regenerate it from its seed - `info.seed` in "
            "a downloaded replay, since `configuration.seed` is usually null - together with "
            "the recorded actions and a matching engine version.")
    farm = deepcopy(farm)
    private = deepcopy(private)
    action = action if isinstance(action, dict) else {}
    hands = action.get("hands", [])
    hands = hands if isinstance(hands, list) else []
    actions = [action.get("farmer", ["PASS"]), *hands]

    # The interpreter rejects all PLANT requests for a crop if seed demand
    # exceeds the available stock, before applying any unit actions.
    demand = defaultdict(int)
    for act in actions:
        if isinstance(act, list) and len(act) >= 2 and act[0] == "PLANT":
            demand[act[1]] += 1
    blocked = {crop for crop, count in demand.items()
               if count > private["seeds"].get(crop, 0)}
    for idx, act in enumerate(actions):
        if (isinstance(act, list) and len(act) >= 2 and act[0] == "PLANT"
                and act[1] in blocked):
            act = ["PASS"]
        _apply_unit_action(farm, private, idx, act, board_size, obs["day"],
                           turns_per_day, shed_capacity)
    return private["shed"]


def _queues(cur, max_orders=10):
    """Both players' market order lists for a turn, truncated the way the engine does."""
    out = []
    for p in (0, 1):
        cell = cur[p] if isinstance(cur[p], dict) else {}
        act = cell.get("action")
        m = act.get("market") if isinstance(act, dict) else None
        out.append(list(m)[:max_orders] if isinstance(m, list) else [])
    return out


def autopsy(ep):
    """Per-turn attribution for both players, with an explicit accuracy report."""
    steps = ep["steps"]
    cfg = ep.get("configuration") or {}
    params = resolve_market_params(cfg)
    max_orders = int(cfg.get("maxMarketOrdersPerTurn") or 10)
    capacity = int(cfg.get("shedCapacity") or 100)
    turns_per_day = max(1, int(cfg.get("turnsPerDay", 24)))
    board_size = int(cfg.get("boardSize") or 10)
    hire_mult = cfg.get("farmHandCostMult")
    hire_mult = 1 if hire_mult is None else hire_mult

    rows = []
    signed, absolute, worst = [0.0, 0.0], [0.0, 0.0], [0.0, 0.0]
    mismatched = 0

    for t in range(1, len(steps)):
        prev, cur = steps[t - 1], steps[t]
        o = prev[0]["observation"]
        money_pre = [o["farms"][p]["money"] for p in (0, 1)]
        money_obs = [cur[0]["observation"]["farms"][p]["money"] for p in (0, 1)]
        hires = [o["farms"][p]["hires_today"] for p in (0, 1)]
        quads = [len(o["farms"][p]["unlocked_quadrants"]) for p in (0, 1)]
        queues = _queues(cur, max_orders)
        sheds = [shed_before_market(
            prev[p], (cur[p].get("action") if isinstance(cur[p], dict) else None),
            p, board_size, capacity, turns_per_day) for p in (0, 1)]

        led, _, after, _ = replay_market_turn(
            queues, o["market"]["inventory"], money_pre, hires, quads, sheds,
            params, capacity, hire_mult)

        # Use the action day; the next observation may already be on the next day.
        day = o["day"]
        for p in (0, 1):
            gap = money_obs[p] - after[p]
            if abs(gap) > 1e-6:
                mismatched += 1
                signed[p] += gap
                absolute[p] += abs(gap)
                worst[p] = max(worst[p], abs(gap))
            for cat, item, u, cash in led[p]:
                rows.append({"step": t, "day": day, "player": p, "category": cat,
                             "item": item, "units": u, "cash": cash})

    report = {
        "player_turns": 2 * (len(steps) - 1),
        "mismatched_player_turns": mismatched,
        "signed": signed,          # Opposite errors can cancel.
        "absolute": absolute,      # Sum of absolute errors across turns.
        "worst_turn": worst,
    }
    return pd.DataFrame(rows, columns=LEDGER_COLUMNS), report


def analyse(ep):
    """Collect the ledger, checks, and plotting inputs for one episode."""
    cfg = ep.get("configuration") or {}
    steps = ep["steps"]
    turns_per_day = int(cfg.get("turnsPerDay") or 24)
    ledger, report = autopsy(ep)
    params = resolve_market_params(cfg)
    return {
        "episode": ep, "steps": steps, "config": cfg, "params": params,
        "ledger": ledger, "report": report,
        "turns_per_day": turns_per_day,
        "last_day": (len(steps) - 1) / turns_per_day,
        "start": [steps[0][0]["observation"]["farms"][p]["money"] for p in (0, 1)],
        "final": [steps[-1][0]["observation"]["farms"][p]["money"] for p in (0, 1)],
    }


A = analyse(episode)
report = A["report"]

print(f"{len(A['ledger']):,} reconstructed ledger entries over {report['player_turns']:,} player-turns")
for p in (0, 1):
    print(f"  player {p}: signed ${report['signed'][p]:>10,.2f}"
          f"   absolute ${report['absolute'][p]:>10,.2f}"
          f"   worst single turn ${report['worst_turn'][p]:>8,.2f}")
print(f"  player-turns that did not reconcile: {report['mismatched_player_turns']}")

assert report["mismatched_player_turns"] == 0, (
    "some turns did not reconcile - read the report above before trusting the P&L")

For this demo, every player's reconstructed **net cash change per turn** matches the
recorded change. That is a useful consistency check, not proof that every individual
trade is correct: offsetting errors within a turn can still escape this check.

Interpreting the categories also requires care. For example, the `feed` bucket records
wheat purchases, including any wheat later resold rather than fed to an animal.

In [ ]:
CAT_ORDER = ["revenue", "seeds", "livestock", "feed", "fertilizer", "labor", "land"]


def pnl(ledger, player=0, params=MARKET_PARAMS):
    """Summarise cash flows using the episode's resolved market parameters."""
    df = ledger[ledger.player == player]
    t = (df.groupby(["category", "item"], as_index=False)
           .agg(units=("units", "sum"), cash=("cash", "sum")))
    base = t["item"].map(lambda i: params[i]["base"] if i in params else None)
    is_rev = t.category.eq("revenue")
    t["avg price"] = (t.cash / t.units).where(is_rev).round(1)
    t["base price"] = base.where(is_rev)
    t["revenue gap vs base"] = (t.units * base - t.cash).where(is_rev).round(0)
    t["_order"] = t.category.map(lambda c: CAT_ORDER.index(c) if c in CAT_ORDER else 99)
    return (t.sort_values(["_order", "cash"], ascending=[True, False])
             .drop(columns="_order").reset_index(drop=True))


table = pnl(A["ledger"], 0, A["params"])
start = A["start"][0]
gross = table.loc[table.category.eq("revenue"), "cash"].sum()
costs = table.loc[~table.category.eq("revenue"), "cash"].sum()
print(f"start ${start:,.0f}   + revenue ${gross:,.0f}   - costs ${-costs:,.0f}"
      f"   = ${start + gross + costs:,.0f}\n")
table.fillna("")

The table lists sales revenue first, followed by spending categories.

`revenue gap vs base` equals units sold times base price, minus realized revenue.
A positive value means sales earned less than the base-price reference; a negative
value means they earned more. Both players' trades and town consumption affect this
gap, so it does not isolate the effect of your own selling.

In [ ]:
def cash_curve(ep, player):
    return [st[0]["observation"]["farms"][player]["money"] for st in ep["steps"]]


fig, ax = plt.subplots(figsize=(8, 3.4))
for player, colour, name in ((0, BLUE, "player 0"), (1, ORANGE, "player 1")):
    series = cash_curve(A["episode"], player)
    ax.plot([s / A["turns_per_day"] for s in range(len(series))], series,
            lw=2, color=colour, label=name)
    ax.annotate(f"{name}  ${series[-1]:,.0f}", xy=(A["last_day"], series[-1]), xytext=(6, 0),
                textcoords="offset points", va="center", color=colour,
                fontsize=8.5, weight="bold")

ax.axhline(A["start"][0], lw=1, color=GRID, zorder=0)
ax.set_xlim(0, max(1, A["last_day"] * 1.22))
ax.set_xlabel("day")
ax.set_ylabel("cash balance")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
ax.grid(axis="y")
ax.set_title("Cash balance through the season")
plt.show()

## 6. Revenue and spending over the season

The daily chart shows when player 0 received sales revenue and paid expenses. The next
chart groups spending by category and gives total revenue in the title for context.

In [ ]:
p0 = A["ledger"][A["ledger"].player == 0]
daily = (p0.assign(kind=p0.category.map(lambda c: "revenue" if c == "revenue" else "cost"))
           .pivot_table(index="day", columns="kind", values="cash", aggfunc="sum")
           .reindex(range(int(A["last_day"]) + 1)).fillna(0.0))

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(daily.index, daily.get("revenue", 0.0), width=0.72, color=BLUE, label="sales revenue")
ax.bar(daily.index, daily.get("cost", 0.0), width=0.72, color=RED, label="spending")
ax.axhline(0, lw=1, color=INK_2)
ax.set_xlabel("day")
ax.set_ylabel("cash flow")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
ax.grid(axis="y")
ax.legend(loc="upper left", ncols=2)
ax.set_title("Daily cash flow: when the farm was buying, and when it was earning")
plt.show()

In [ ]:
p0 = A["ledger"][A["ledger"].player == 0]
gross = p0.loc[p0.category.eq("revenue"), "cash"].sum()
buckets = (p0[~p0.category.eq("revenue")].groupby("category")["cash"].sum().abs()
             .sort_values(ascending=True))

fig, ax = plt.subplots(figsize=(8, 3.0))
ax.barh(list(buckets.index), list(buckets.values), height=0.62, color=RED)
for name, value in buckets.items():
    ax.annotate(f"${value:,.0f}", xy=(value, name), xytext=(6, 0), textcoords="offset points",
                va="center", fontsize=8.5, color=INK_2)
ax.set_xlim(0, max(1, buckets.max() * 1.25) if not buckets.empty else 1)
ax.set_xlabel("total spend")
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
ax.grid(axis="x")
ax.set_title(f"Costs against ${gross:,.0f} of gross revenue")
plt.show()

## 7. Prices as supply accumulates

These charts use the installed engine's **default parameters**, independently of `A`.
For each product, we start at `I0` and calculate the price of successive units sold.

The example assumes one seller, no town consumption, and unlimited shed capacity.
It illustrates the price curve rather than an executable sales schedule. It also
excludes production costs, so it does not measure profit or identify an optimal volume.

In [ ]:
def marginal_curve(item, n_max, params=MARKET_PARAMS):
    """Return successive sell quotes from I0, with no other market activity.

    Ignore shed capacity to illustrate the curve beyond feasible single-turn sales.
    """
    inv, out = params[item]["I0"], []
    for _ in range(n_max):
        price = price_of(item, inv, params)
        out.append(price)
        if price > PRICE_FLOOR:
            inv += 1
    return out


order = ["WHEAT", "CARROT", "TOMATO", "EGG", "FERTILIZER", "STRAWBERRY", "MILK", "WOOL", "MELON"]
fig, axes = plt.subplots(3, 3, figsize=(8.6, 6.6), sharex=True)
for ax, item in zip(axes.ravel(), order):
    base = MARKET_PARAMS[item]["base"]
    curve = marginal_curve(item, 300)
    ax.plot(range(1, 301), curve, lw=2, color=BLUE)
    ax.axhline(base, lw=1, ls=(0, (3, 3)), color=GRID)
    half = next((i + 1 for i, p in enumerate(curve) if p <= base / 2), None)
    if half:
        ax.axvline(half, lw=1, color=ORANGE)
        ax.annotate(f"half price\nat {half} units", xy=(half, base * 0.60), xytext=(6, 0),
                    textcoords="offset points", fontsize=7.5, color=ORANGE, weight="bold")
    ax.set_title(f"{item.title()}   base ${base}", fontsize=9.5)
    ax.set_ylim(0, base * 1.15)
    ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
    ax.grid(axis="y")
axes[-1][1].set_xlabel("cumulative units sold into the market")
fig.suptitle("What each extra unit fetches as supply accumulates",
             x=0.005, ha="left", fontsize=11, weight="bold", color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

In [ ]:
cliff = pd.DataFrame([{
    "product": item,
    "base": MARKET_PARAMS[item]["base"],
    "units to half price": next((i + 1 for i, p in enumerate(marginal_curve(item, 1500))
                                 if p <= MARKET_PARAMS[item]["base"] / 2), None),
    "units to $1": next((i + 1 for i, p in enumerate(marginal_curve(item, 1500)) if p <= 1), None),
} for item in order]).sort_values("units to half price").reset_index(drop=True)
cliff.fillna("> 1500")

## 8. Run it on your own episodes

Use a replay with recorded actions, public farm state, and full private observations
for both players. Missing private fields raise an error because shed contents cannot
be reconstructed reliably without them.

Choose one of these inputs after running the function definitions above:

```python
# A downloaded replay attached as a dataset.
import json
with open("/kaggle/input/<your-dataset>/<episode>.json") as fh:
    A = analyse(json.load(fh))

# Alternatively, generate an episode locally.
env = make("kaggriculture", configuration={"seed": 7})
env.run(["/kaggle/working/my_agent.py", "starter"])
A = analyse(env.toJSON())

print(A["report"])
pnl(A["ledger"], player=0, params=A["params"])
```

After replacing `A`, rerun the P&L cell and the cash and spending charts in sections
5 and 6. Each chart obtains its episode data from `A`; the player labels are seat
numbers. Section 3 remains a summary of the original demo, and sections 1 and 7 show
default price curves.

**Check `A["report"]` before interpreting the tables.** Zero mismatched player-turns
means the reconstructed net cash changes match the recorded ones. It does not certify
every individual trade. Nonzero `absolute` or `worst_turn` values indicate a discrepancy
to investigate, including possible engine-version differences or a replay-model error.

**Version and data requirements.** The episode must carry both players' full private
state and the tile grid at every step; unit actions are replayed against the farm, so a
replay trimmed to save space raises an error rather than guessing. If your stored copy
drops the grid, regenerate the episode from its seed (`info.seed` in a downloaded replay;
`configuration.seed` is usually null), the same configuration, the recorded actions, and a
matching engine version, then check the regenerated observations against the fields you
kept. This notebook is
tested with `kaggle-environments==1.32.7`. It reads the episode's market, capacity, hiring, board,
and timing settings, but uses the installed engine's unit-action helper, price function,
and cost constants. Replays from other engine versions may behave differently. The
unit-action helper is an internal API, so upgrading the package requires revalidation.

## What the tables leave out

The ledger records cash, which determines the final score. It does not value unsold
inventory or measure the return an alternative plan could have earned. Seed, land,
and animal purchases appear as cash outflows when paid; these tables do not allocate
their costs across individual products or days of use.

Idle land and missed production therefore need separate investigation. Start with the
reconciliation report, inspect spending and realized prices, and use the replay to
understand the actions behind those numbers.